In [ ]:
import pandas as pd
import os

# list of local CSV paths
files = [
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_SI.csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_CH.csv", 
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_CB.csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_CT.csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_SL.csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Profile_FB.csv",
]

# read and concatenate with pitch type from filename
profile_dfs = []
for path in files:
    df = pd.read_csv(path)
    # Extract pitch type from last 2 chars of filename before .csv
    pitchtype = os.path.basename(path)[-6:-4]  
    df['pitchtype'] = pitchtype
    profile_dfs.append(df)

pitch_profile = pd.concat(profile_dfs, ignore_index=True)

# Add unique pitchuid for each row
pitch_profile['pitchuid'] = range(len(pitch_profile))

# now append to your existing df (assuming matching columns)
df = pitch_profile
df.head()


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    # keep only needed cols (including handedness)
    needed_cols = [
        "pitcher","relside","relspeed","spinrate","extension",
        "relheight","horzbreak","inducedvertbreak","pitchtype",
        "pitchuid","throwshand"
    ]
    df = df[needed_cols].copy()

    # use existing throwsHand for handedness
    df["pitcher_hand"] = df["throwshand"].str.upper()

    # rename to model‐expected names
    df = df.rename(columns={
        "relspeed":         "start_speed",
        "spinrate":         "spin_rate",
        "extension":        "extension",
        "relheight":        "z0",
        "relside":          "x0",
        "horzbreak":        "ax",
        "inducedvertbreak": "az",
        "pitchtype":        "pitch_type"
    })

    # ensure numeric before mirroring
    df[["ax","x0","z0","start_speed","spin_rate","extension"]] = (
        df[["ax","x0","z0","start_speed","spin_rate","extension"]]
        .apply(pd.to_numeric, errors="coerce")
    )

    # mirror for LHP
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

    # fastball logic
    fastball_types = ["Four-Seam","Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)]
    df_agg = (
        df_fb.groupby(["pitcher","pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed","mean"),
                 avg_fastball_az=("az","mean"),
                 avg_fastball_ax=("ax","mean"),
                 count=("start_speed","count")
             )
             .sort_values(["count","avg_fastball_speed"], ascending=[False,False])
             .drop_duplicates(subset=["pitcher"], keep="first")
    )
    df = df.merge(
        df_agg[["pitcher","avg_fastball_speed","avg_fastball_az","avg_fastball_ax"]],
        on="pitcher", how="left"
    )

    # diffs & flags
    df["speed_diff"]   = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]      = df["az"]          - df["avg_fastball_az"]
    df["ax_diff"]      = df["ax"]          - df["avg_fastball_ax"]
    df["is_fastball"]  = df["pitch_type"].isin(fastball_types)

    # inches conversion
    df["z0"] *= 12
    df["x0"] *= 12

    return df

def run_model_and_scale(df: pd.DataFrame) -> pd.DataFrame:
    # load model
    model_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_ALL (1).joblib"
    model = joblib.load(model_path)

    feats = [
        "start_speed","spin_rate","extension",
        "az","ax","x0","z0",
        "speed_diff","az_diff","ax_diff","is_fastball"
    ]
    df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")

    # predict & scale
    df["target"] = model.predict(df[feats])
    mean2023, std2023 = 0.011532333993710725, 0.009399038486978739
    df["target_zscore"]    = (df["target"] - mean2023) / std2023
    df["tj_stuff_plus"]    = 100 - (df["target_zscore"] * 10)

    return df

# --- MAIN SCRIPT ---

# rename raw columns
df = df.rename(columns={
    "player":        "pitcher",
    "Vel":           "relspeed",
    "Spin":          "spinrate",
    "Extension":     "extension",
    "RelX":          "relside",
    "RelZ":          "relheight",
    "HorzBrk":       "horzbreak",
    "IndVertBrk":    "inducedvertbreak",
    "Pitchtype":     "pitchtype",
    "throwsHand":    "throwshand"
})
df.columns = df.columns.str.lower()

# run feature engineering + model
df_fe = feature_engineering(df.copy())
df_fe = run_model_and_scale(df_fe)

# merge back tj_stuff_plus
df = df.merge(
    df_fe[["pitchuid","tj_stuff_plus"]],
    on="pitchuid", how="left"
).drop(columns=["pitchuid"])


In [ ]:
df.sort_values('tj_stuff_plus', ascending=False).head()

In [ ]:
# Get subset of columns and pivot pitchtypes before exporting
stuff_df = df[['playerid', 'playerfullname', 'pitchtype', 'tj_stuff_plus']]
pivoted_df = stuff_df.pivot(index=['playerid', 'playerfullname'], 
                          columns='pitchtype',
                          values='tj_stuff_plus')
pivoted_df.to_csv('C:/Users/TrevorWhite/Downloads/transfer_stuff_plus_analysis.csv')

In [ ]:
seperately calculate percentiles for relside	relheight 	horzrelangle vertrelangle	vertapprangle	horzapprangle

In [ ]:
import pandas as pd

# List of CSV file paths to append
csv_files = [
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (17).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (16).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (15).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (14).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (13).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (12).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (11).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (10).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (9).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (8).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (20).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (19).csv",
    r"C:\Users\TrevorWhite\Downloads\Pitch Metrics (18).csv",
    r"C:\Users\TrevorWhite\Downloads\p_guys.csv"
]

# Read and concatenate all CSVs
dfs = [pd.read_csv(f) for f in csv_files]
all_df = pd.concat(dfs, ignore_index=True)

# Drop duplicates, keeping the first occurrence
deduped_df = all_df.drop_duplicates(keep='first')

# Save back to p_guys.csv
deduped_df.to_csv(r"C:\Users\TrevorWhite\Downloads\p_guys.csv", index=False)

In [ ]:
# Append USD pitching year-to-date data to p_guys.csv

import pandas as pd

# Read the existing p_guys.csv file
p_guys_path = r"C:\Users\TrevorWhite\Downloads\p_guys.csv"
p_guys_df = pd.read_csv(p_guys_path)

# Read the USD pitching year-to-date data
usd_pitching_path = r"C:\Users\TrevorWhite\Downloads\USDPITCHINGYTD.csv"
usd_pitching_df = pd.read_csv(usd_pitching_path)

# Concatenate the two DataFrames
combined_df = pd.concat([p_guys_df, usd_pitching_df], ignore_index=True)

# Drop duplicates, keeping the first occurrence
combined_df = combined_df.drop_duplicates(keep='first')

# Save the combined DataFrame back to p_guys.csv
# combined_df.to_csv(p_guys_path, index=False)
# Save the combined DataFrame back to USDPITCHINGYTD.csv as well
combined_df.to_csv(usd_pitching_path, index=False)
